# `evap_cool` — multi-trap dimensionless overview

This notebook builds the **single-page dimensionless overlay figure** that
replaces the six per-trap dimensional plots (`plot_state_functions` ×3 +
`plot_thermal_coefficients` ×3) with one figure capturing the universal
classical-vs-quantum thermodynamics across the box, oscillator, and
quadrupole traps.

For every trap and statistic that has both a source run (`*_<stat>.json`)
and its post-processed sibling (`*_<stat>_thermo.json`) in the latest
session, the notebook normalises each dimensional quantity per step by
the natural scale ($N k_B T$ for energies, $N k_B$ for $C_V$, $C_P$,
$S$; $V_g/(N k_B T)$ for $\kappa_T$; $T$ for $B_P$) and overlays all
three traps on the same panels.

> **Prerequisites.** Run `evap_cool_usage.ipynb` first to produce the
> source `*.json` files, then `evap_cool_thermo_usage.ipynb` to produce
> the `*_thermo.json` siblings. Missing files are skipped silently —
> if a session only has BE and FD thermo (no MB), the figure renders
> without MB lines but with the per-trap MB asymptotes still shown as
> faint horizontal references.


## Setup

We import the new plotter from `evap_cool.plots`:

- `plot_dimensionless_overview(traps, *, panels=...)` — pass a list of
  trap dicts, get back a Matplotlib figure.

If the import below fails with `ImportError`, the additions in
`plots_additions.py` haven't been merged into `evap_cool/plots.py` yet
(and `plot_dimensionless_overview` hasn't been re-exported from
`evap_cool/__init__.py`). See the integration notes in
`plots_additions.py` for how to land it.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from evap_cool import (
    load_run, load_thermodynamics,
    list_sessions, list_runs,
    BoxTrap, OscillatorTrap, QuadrupoleTrap,
    plot_dimensionless_overview,
)


## Locate the most recent session

Same convention as `evap_cool_thermo_usage.ipynb`: pick the latest
`runs/<date>/<time>/` folder. Override `session` manually if you want
to plot an older run.


In [ ]:
sessions = list_sessions("runs")
if not sessions:
    raise RuntimeError(
        "No sessions found under runs/. "
        "Run evap_cool_usage.ipynb and evap_cool_thermo_usage.ipynb first."
    )

session = sessions[-1]
print(f"Using session : {session}")
print(f"Files present :")
for p in list_runs(session):
    print(f"  {p.relative_to(session)}")


## Reconstruct trap instances

The dimensionless normalisation needs three trap-specific scalars: the
density-of-states exponent $s$, the generalised volume $V_g$, and the
Boltzmann constant $k_B$ in the trap's unit system (SI for the box,
eV for the others). All three are stored on the `Trap` instance, so
we rebuild the same trap objects used at run time and read them off.

If you customised any trap parameter in `evap_cool_usage.ipynb`,
mirror the change here so $V_g$ comes out right.


In [ ]:
trap_box  = BoxTrap(V=1e-11)
trap_osc  = OscillatorTrap(omega=2 * np.pi * 100)
trap_quad = QuadrupoleTrap(A_bar=1e-15)

for label, trap in (("box", trap_box),
                    ("oscillator", trap_osc),
                    ("quadrupole", trap_quad)):
    print(f"  {label:10s}  s = {trap.s},  V_g = {trap.volume_global:.4g},  "
          f"kB = {trap.kB:.4g}")


## Helper: assemble one trap's payload

For each trap we walk through the three possible statistics
(`bosons`, `fermions`, `mb`) and load the (source, thermo) pair if both
files exist. The returned dict has the shape
`plot_dimensionless_overview` expects:

```python
{
    "name":  ...,  "s": ..., "V_g": ..., "kB": ...,
    "bosons":   {"T": [...], "N": [...], "thermo": {...}},   # optional
    "fermions": {...},                                          # optional
    "mb":       {...},                                          # optional
}
```

Missing statistics are skipped silently; the plotter handles
the omission by drawing fewer curves on each panel.


In [ ]:
def build_trap_entry(name, trap, label, session):
    """Load all available (source, thermo) pairs for one trap."""
    entry = {
        "name": name,
        "s":    trap.s,
        "V_g":  trap.volume_global,
        "kB":   trap.kB,
    }
    found = []
    for stat_key, file_suffix in (("bosons",   "bosons"),
                                  ("fermions", "fermions"),
                                  ("mb",       "mb")):
        src_path = session / f"{label}_{file_suffix}.json"
        thr_path = session / f"{label}_{file_suffix}_thermo.json"
        if not (src_path.exists() and thr_path.exists()):
            continue
        src = load_run(src_path)["results"]
        thr = load_thermodynamics(thr_path)["results"]
        entry[stat_key] = {"T": src["T"], "N": src["N"], "thermo": thr}
        found.append(stat_key)
    return entry, found


## Build the `traps` list

Loop over the three traps, calling the helper above for each. Any trap
with zero loaded statistics (e.g. post-processing hasn't run yet) is
dropped with a printed note.


In [ ]:
traps = []
for name, trap, label in [
    ("Box",        trap_box,  "box"),
    ("Oscillator", trap_osc,  "oscillator"),
    ("Quadrupole", trap_quad, "quadrupole"),
]:
    entry, found = build_trap_entry(name, trap, label, session)
    if not found:
        print(f"  [skip] {name}: no (source, thermo) pairs found")
        continue
    print(f"  {name:10s}  loaded: {', '.join(found)}")
    traps.append(entry)

if not traps:
    raise RuntimeError(
        "No trap data was loaded. "
        "Did evap_cool_thermo_usage.ipynb finish successfully?"
    )


## 1. Headline figure (2×2 — main paper)

Four panels:

- **$C_V / (N k_B)$** — encodes the degrees-of-freedom story directly:
  the three MB plateaus sit at $s = 3/2,\,3,\,9/2$ (the trap colors).
  BE flares up just below condensation; FD is suppressed by Pauli
  blocking.
- **$|\Omega| / (N k_B T) = P V_g / (N k_B T)$** — the generalised
  ideal-gas relation. MB sits at $1$; the BE/FD splay is the
  bosonic-enhancement / fermionic-pressure signature.
- **$\kappa_T \, N k_B T / V_g$** — dimensionless compressibility.
  MB at $1$; BE diverges near BEC; FD saturates well below $1$.
- **$B_P \cdot T$** — dimensionless thermal expansion, same regime
  behaviour as $\kappa_T$.

Color = trap, line style = statistic (solid MB, dashed BE, dotted FD).


In [ ]:
fig_headline = plot_dimensionless_overview(traps, panels="headline")
plt.show()


## 2. Full figure (2×4 — supplementary)

Same overlay, eight panels. Adds $S / (N k_B)$, $H / (N k_B T)$,
$C_P / (N k_B)$, and $P V_g / (N k_B T)$ (the latter is the same as
$|\Omega| / (N k_B T)$ but useful pedagogically alongside the
heat capacities). Useful if a reviewer asks for the complete
dimensionless set.


In [ ]:
fig_full = plot_dimensionless_overview(traps, panels="full")
plt.show()


## 3. Save the figures

PDF for the paper, PNG for the supplementary website / arXiv ancillary.
Files land in `session/figures/`, alongside the JSON runs they came from.


In [ ]:
out_dir = session / "figures"
out_dir.mkdir(exist_ok=True)

for layout in ("headline", "full"):
    fig = plot_dimensionless_overview(traps, panels=layout)
    pdf_path = out_dir / f"dimensionless_{layout}.pdf"
    png_path = out_dir / f"dimensionless_{layout}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {pdf_path.relative_to(session)} and *.png")


## 4. Reading the figures

A few things worth knowing when reading these panels:

- **Color = trap, line style = statistic.** Box (blue), Oscillator
  (green), Quadrupole (red); solid MB, dashed BE, dotted FD. The
  legend is split into two blocks below the grid so it stays
  readable with up to nine curves per panel.

- **Faint horizontal lines = MB high-$T$ asymptotes.** On panels where
  the MB limit is a constant function of $s$ ($C_V$, $C_P$, $H$,
  $|\Omega|$, $\kappa_T$, $B_P\cdot T$), each trap's MB plateau is
  drawn faintly even if no MB run was loaded. They act as a
  legend-free reminder of which $s$ each color corresponds to.

- **Curves converge at high $T$.** All three statistics reduce to MB
  in the non-degenerate limit, where the polylog ratios collapse to
  unity. The high-$T$ part of each panel is where you check the
  classical limit; the low-$T$ part is where the statistics diverge.

- **BE curves stop short of the lowest $T$** because the run halts as
  the system approaches BEC. The colder endpoint of each dashed
  curve is the closest the cooling protocol got to degeneracy before
  $\mu \to 0^-$ made the rethermalisation step fail. FD curves
  continue smoothly because the fermionic system never crosses a
  phase transition — it just becomes increasingly Sommerfeld-dominated.

- **The quadrupole (red) often stays near its MB plateau across the
  full range.** This isn't a numerical issue: it's the paper's
  central point that $s = 9/2$ requires lower temperatures (or higher
  densities) to reach degeneracy than $s = 3/2$ or $s = 3$. The fact
  that the red dashed / dotted curves stay close to the red solid
  line *is* the result.
